In [ ]:
from pathlib import Path
import struct
import re
from typing import BinaryIO

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# FILE = Path("/Users/sylvi/Downloads/example_files/Ankita_example_file.pfc")
FILE = Path("/Users/sylvi/topo_data/paloma/QNM_data/cell3.0_00000.pfc")

In [ ]:
def read_float(open_file: BinaryIO) -> float:
    """Read a float from an open binary file.

    Parameters
    ----------
    open_file: BinaryIO
        An open binary file object.

    Returns
    -------
    float
    Float decoded value.
    """
    return struct.unpack("f", open_file.read(4))[0]


def read_double(open_file: BinaryIO) -> float:
    """Read an 8 byte double from an open binary file.

    Parameters
    ----------
    open_file: BinaryIO
        An open binary file object.

    Returns
    -------
    float
        Float decoded from the double value
    """
    return struct.unpack("d", open_file.read(8))[0]


def read_int32(open_file: BinaryIO) -> int:
    """Read a signed 32 bit integer from an open binary file.

    Parameters
    ----------
    open_file: BinaryIO
        An open binary file object.

    Returns
    -------
    int
        Integer decoded value.
    """
    return struct.unpack("i", open_file.read(4))[0]

In [ ]:
with open(FILE, "rb") as f:
    # Calculate file length
    file_length = f.seek(0, 2)
    print(f"file length: {file_length} bytes")
    f.seek(0)

    # Read the metadata
    metadata = b""
    end_of_metadata = False
    while not end_of_metadata:
        byte = f.read(1)
        if byte == b"\x1a":
            end_of_metadata = True
        metadata += byte

    print(f"end of metadata position: {f.tell()}")
    metadata = metadata.split(b"\r\n")
    # print("metadata:")
    # for line in metadata:
    #     print(f" {line}")

    # Find the data
    blank_section_beginning_pos = f.tell()
    print(f"Searching for non-null byte, starting at position {blank_section_beginning_pos}")
    byte = f.read(1)
    index = 0
    broken = False
    while byte == b"\x00":
        byte = f.read(1)
        index += 1
        if index > 100000:
            broken = True
            break
    #         print(f"index: {index}, position in file: {f.tell()}")
    if broken:
        print("broken")
    else:
        print(
            f"found byte {byte} at byte position {f.tell()}, {f.tell() - blank_section_beginning_pos} past the metadata"
        )

    # nums = []
    # for i in range(100000):
    #     nums.append(read_int(f))

    # nums = np.array(nums)

    # print(nums[nums != 0.0])

    # f.seek(80900)
    # bytes = f.read(200)
    # print(bytes)

    # data = f.read(10000)
    # print(data)

    # Offset of 3 bytes after skipping the null values gives all positive values in region of 12800 or positive floats of magnitude e-34

    # offset = 3
    # f.read(offset)

    f.seek(-20, 2)

    print(f.read(20))

    f.seek(-4, 2)
    print(read_int32(f))
    f.seek(-8, 2)
    print(read_int32(f))

    skip = 4
    end_nums = []
    f.seek(-skip, 2)
    for i in range(160000):
        num = read_int32(f)
        f.seek(-(i + 1) * skip, 2)
        # print(f.tell())
        end_nums.append(num)

    end_nums.reverse()

    batch_size = 40000
    num_batches = int(np.ceil(len(end_nums) / batch_size))
    for i in range(num_batches):
        batch = end_nums[i * batch_size : i * batch_size + batch_size]
        plt.plot(batch)
        plt.show()

    # for i in range(10000000):
    #     num = read_float(f)
    #     if num > 0.1:
    #         print(num)
    #         break